# SARSA: Latihan Interaktif

Notebook ini berisi latihan hands-on untuk memahami SARSA dan perbedaannya dengan Q-Learning.

Lihat `README.md` untuk teori lengkap tentang on-policy vs off-policy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys

sys.path.append('../01_q_learning')
from implementation import GridWorldSPG, QLearningAgent

sys.path.append('../02_sarsa')
from implementation import SARSAAgent, compare_sarsa_vs_qlearning, plot_training_comparison, visualize_policy

np.random.seed(42)
print("✓ Setup complete!")

## Latihan 1: Basic SARSA Training

In [ ]:
# Create SPG grid-world environment
env = GridWorldSPG(grid_size=5, n_outlets=3, n_obstacles=3, seed=42)

print("Environment:")
print(f"  Grid size: {env.grid_size}x{env.grid_size}")
print(f"  Outlets: {env.n_outlets}")
print(f"  Obstacles: {env.n_obstacles}")

# Create and train SARSA agent
sarsa_agent = SARSAAgent(env, learning_rate=0.1, discount_factor=0.95, seed=42)
print("\nTraining SARSA agent...")
history = sarsa_agent.train(n_episodes=500, verbose=True)

# Evaluate
avg_reward, avg_steps = sarsa_agent.evaluate(n_episodes=100)
print(f"\nEvaluation Results:")
print(f"  Average Reward: {avg_reward:.2f}")
print(f"  Average Steps: {avg_steps:.1f}")

## Latihan 2: Visualize Learning Curve

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rewards
ax = axes[0]
window = 20
rewards = history['episode_rewards']
smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
ax.plot(smoothed, linewidth=2)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Average Reward', fontsize=12)
ax.set_title('SARSA Learning Curve (Rewards)', fontweight='bold')
ax.grid(True, alpha=0.3)

# Epsilon decay
ax = axes[1]
ax.plot(history['epsilon_values'], linewidth=2, color='orange')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Epsilon', fontsize=12)
ax.set_title('Epsilon Decay', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observation: Epsilon decays → more exploitation over time")

## Latihan 3: Visualize Learned Policy

In [ ]:
# Show learned policy
fig = visualize_policy(sarsa_agent, "SARSA Learned Policy")
plt.show()

print("Policy shows optimal actions (arrows) for each state.")
print("Agent learns to navigate around obstacles to reach outlets.")

## Latihan 4: SARSA vs Q-Learning Comparison

In [ ]:
# Create fresh environment
env_compare = GridWorldSPG(grid_size=5, n_outlets=3, n_obstacles=3, seed=42)

# Train both agents
print("Training SARSA...")
sarsa = SARSAAgent(env_compare, seed=42)
sarsa.train(n_episodes=500, verbose=False)

print("Training Q-Learning...")
qlearning = QLearningAgent(env_compare, seed=42)
qlearning.train(n_episodes=500, verbose=False)

# Evaluate both
sarsa_reward, sarsa_steps = sarsa.evaluate(100)
qlearning_reward, qlearning_steps = qlearning.evaluate(100)

print("\nEvaluation Results:")
print(f"SARSA:      Reward={sarsa_reward:.2f}, Steps={sarsa_steps:.1f}")
print(f"Q-Learning: Reward={qlearning_reward:.2f}, Steps={qlearning_steps:.1f}")

# Plot comparison
fig = plot_training_comparison(sarsa, qlearning)
plt.show()

## Latihan 5: Parameter Sensitivity - Learning Rate

In [ ]:
# Test different learning rates
learning_rates = [0.01, 0.1, 0.3, 0.5]
results = []

for lr in learning_rates:
    env_test = GridWorldSPG(grid_size=5, n_outlets=3, n_obstacles=3, seed=42)
    agent = SARSAAgent(env_test, learning_rate=lr, seed=42)
    agent.train(n_episodes=500, verbose=False)
    avg_reward, _ = agent.evaluate(100)
    results.append(avg_reward)
    print(f"Learning Rate {lr:.2f}: Avg Reward = {avg_reward:.2f}")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(learning_rates, results, marker='o', linewidth=2, markersize=10)
plt.xlabel('Learning Rate', fontsize=12)
plt.ylabel('Average Reward (Evaluation)', fontsize=12)
plt.title('SARSA Performance vs Learning Rate', fontweight='bold', fontsize=13)
plt.grid(True, alpha=0.3)
plt.show()

best_lr = learning_rates[np.argmax(results)]
print(f"\nBest learning rate: {best_lr}")

## Latihan 6: Parameter Sensitivity - Discount Factor

In [ ]:
# Test different discount factors
discount_factors = [0.5, 0.7, 0.9, 0.95, 0.99]
results = []

for gamma in discount_factors:
    env_test = GridWorldSPG(grid_size=5, n_outlets=3, n_obstacles=3, seed=42)
    agent = SARSAAgent(env_test, discount_factor=gamma, seed=42)
    agent.train(n_episodes=500, verbose=False)
    avg_reward, _ = agent.evaluate(100)
    results.append(avg_reward)
    print(f"Discount Factor {gamma:.2f}: Avg Reward = {avg_reward:.2f}")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(discount_factors, results, marker='o', linewidth=2, markersize=10, color='orange')
plt.xlabel('Discount Factor (γ)', fontsize=12)
plt.ylabel('Average Reward (Evaluation)', fontsize=12)
plt.title('SARSA Performance vs Discount Factor', fontweight='bold', fontsize=13)
plt.grid(True, alpha=0.3)
plt.show()

print("\nObservation: Higher γ values future rewards more.")

## Latihan 7: On-Policy vs Off-Policy Behavior

This exercise demonstrates the key difference between SARSA and Q-Learning.

In [ ]:
# Compare policies side by side
env1 = GridWorldSPG(grid_size=5, n_outlets=3, n_obstacles=3, seed=42)
env2 = GridWorldSPG(grid_size=5, n_outlets=3, n_obstacles=3, seed=42)

sarsa = SARSAAgent(env1, seed=42)
sarsa.train(n_episodes=800, verbose=False)

qlearning = QLearningAgent(env2, seed=42)
qlearning.train(n_episodes=800, verbose=False)

# Visualize both policies
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# SARSA policy
ax = axes[0]
sarsa_policy = sarsa.get_policy()
grid_size = env1.grid_size

for i in range(grid_size):
    for j in range(grid_size):
        state = (i, j)
        if state == env1.start_pos:
            color = 'lightgreen'
        elif state in env1.outlet_positions:
            color = 'gold'
        elif state in env1.obstacle_positions:
            color = 'lightcoral'
        else:
            color = 'white'
        ax.add_patch(plt.Rectangle((j, grid_size-1-i), 1, 1, facecolor=color, edgecolor='black'))
        
        if state not in env1.obstacle_positions:
            action = sarsa_policy[state]
            arrow = env1.action_names[action]
            ax.text(j+0.5, grid_size-1-i+0.5, arrow, ha='center', va='center', fontsize=18, color='blue')

ax.set_xlim(0, grid_size)
ax.set_ylim(0, grid_size)
ax.set_aspect('equal')
ax.set_title('SARSA (On-Policy)', fontweight='bold', fontsize=14)
ax.grid(True)

# Q-Learning policy
ax = axes[1]
qlearning_policy = qlearning.get_policy()

for i in range(grid_size):
    for j in range(grid_size):
        state = (i, j)
        if state == env2.start_pos:
            color = 'lightgreen'
        elif state in env2.outlet_positions:
            color = 'gold'
        elif state in env2.obstacle_positions:
            color = 'lightcoral'
        else:
            color = 'white'
        ax.add_patch(plt.Rectangle((j, grid_size-1-i), 1, 1, facecolor=color, edgecolor='black'))
        
        if state not in env2.obstacle_positions:
            action = qlearning_policy[state]
            arrow = env2.action_names[action]
            ax.text(j+0.5, grid_size-1-i+0.5, arrow, ha='center', va='center', fontsize=18, color='blue')

ax.set_xlim(0, grid_size)
ax.set_ylim(0, grid_size)
ax.set_aspect('equal')
ax.set_title('Q-Learning (Off-Policy)', fontweight='bold', fontsize=14)
ax.grid(True)

plt.tight_layout()
plt.show()

print("Both algorithms learn effective policies for this environment!")
print("SARSA: Learns from actual behavior (including exploration).")
print("Q-Learning: Learns optimal policy directly.")